In [5]:
import os

print("Current folder:", os.getcwd())
print("\nFiles in ../outputs:")
print(os.listdir("../outputs"))
print("Files in ../dataset:")
print(os.listdir("../dataset"))

Current folder: c:\Users\palak\SANKETSETU\notebooks

Files in ../outputs:
['transformer_history.pkl']
Files in ../dataset:
['processed', 'raw', 'training']


In [7]:
# ============================================
# Transformer V3 - Imports & Load Dataset
# ============================================

import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
    GlobalAveragePooling1D,
    Embedding,
    Add
)

from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

print("TensorFlow Version:", tf.__version__)

# --------------------------------------------
# Load dataset prepared in Day 5
# --------------------------------------------

DATA_PATH = "../dataset/training"

X_train = np.load(os.path.join(DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(DATA_PATH, "y_train.npy"))

X_val = np.load(os.path.join(DATA_PATH, "X_val.npy"))
y_val = np.load(os.path.join(DATA_PATH, "y_val.npy"))

X_test = np.load(os.path.join(DATA_PATH, "X_test.npy"))
y_test = np.load(os.path.join(DATA_PATH, "y_test.npy"))

print("\n✅ Dataset Loaded Successfully!")
print("Train      :", X_train.shape, y_train.shape)
print("Validation :", X_val.shape, y_val.shape)
print("Test       :", X_test.shape, y_test.shape)

num_classes = len(np.unique(y_train))
sequence_length = X_train.shape[1]   # 154 frames
feature_dim = X_train.shape[2]        # 63 features

print("\nClasses         :", num_classes)
print("Sequence Length :", sequence_length)
print("Feature Dim     :", feature_dim)

TensorFlow Version: 2.21.0

✅ Dataset Loaded Successfully!
Train      : (7608, 154, 63) (7608,)
Validation : (275, 154, 63) (275,)
Test       : (487, 154, 63) (487,)

Classes         : 210
Sequence Length : 154
Feature Dim     : 63


In [8]:
# ============================================
# Transformer V3 - Learnable Positional Embedding
# ============================================

EMBED_DIM = 128       # Embedding dimension
NUM_HEADS = 4          # Attention heads
FF_DIM = 256           # Feed-forward network size
NUM_BLOCKS = 3         # Transformer encoder blocks
DROPOUT_RATE = 0.3

# Create position indices: [0, 1, 2, ..., 153]
positions = tf.range(start=0, limit=sequence_length, delta=1)

# Learnable positional embedding layer
position_embedding_layer = Embedding(
    input_dim=sequence_length,
    output_dim=EMBED_DIM,
    name="position_embedding"
)

# Position embeddings shape
position_embeddings = position_embedding_layer(positions)

print("✅ Positional Embedding Ready!")
print("Position Embedding Shape:", position_embeddings.shape)
print("Embedding Dimension:", EMBED_DIM)

✅ Positional Embedding Ready!
Position Embedding Shape: (154, 128)
Embedding Dimension: 128


In [9]:
# ============================================
# Transformer V3 - Encoder Block
# ============================================

def transformer_encoder(inputs, embed_dim, num_heads, ff_dim, dropout=0.3):
    # Multi-Head Self Attention
    attention_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embed_dim // num_heads,
        dropout=dropout
    )(inputs, inputs)

    attention_output = Dropout(dropout)(attention_output)

    # Residual Connection + LayerNorm
    x = Add()([inputs, attention_output])
    x = LayerNormalization(epsilon=1e-6)(x)

    # Feed Forward Network
    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dropout(dropout)(ffn)
    ffn = Dense(embed_dim)(ffn)

    # Residual Connection + LayerNorm
    x = Add()([x, ffn])
    x = LayerNormalization(epsilon=1e-6)(x)

    return x


print("✅ Transformer Encoder Block Created!")

✅ Transformer Encoder Block Created!


In [11]:
# ============================================
# Transformer V3 - Build Model
# ============================================

inputs = Input(shape=(sequence_length, feature_dim), name="gesture_input")

# Project 63 features -> 128-dimensional embedding
x = Dense(EMBED_DIM, name="input_projection")(inputs)

# Add learnable positional embeddings
x = x + position_embeddings

# Stack 3 Transformer Encoder blocks
for i in range(NUM_BLOCKS):
    x = transformer_encoder(
        x,
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        ff_dim=FF_DIM,
        dropout=DROPOUT_RATE
    )

# Global representation
x = GlobalAveragePooling1D()(x)

# Classification head
x = Dropout(0.4)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)

outputs = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=inputs, outputs=outputs, name="Transformer_V3")

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "Transformer_V3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ gesture_input       │ (None, 154, 63)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_projection    │ (None, 154, 128)  │      8,192 │ gesture_input[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 154, 128)  │          0 │ input_projection… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 154, 128)  │     66,048 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 154, 128)  │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 154, 128)  │          0 │ add[0][0],        │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 154, 128)  │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 154, 256)  │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 154, 256)  │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 154, 128)  │     32,896 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 154, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 154, 128)  │     66,048 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 154, 128)  │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 154, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 154, 256)  │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 154, 256)  │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 154, 128)  │     32,896 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 154, 128)  │          0 │ layer_normalizat

 Total params: 492,626 (1.88 MB)

 Trainable params: 492,626 (1.88 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# ============================================
# Transformer V3 - Train Model
# ============================================

import os

os.makedirs("../models", exist_ok=True)

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "../models/best_transformer_v3.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

Epoch 1/60
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 552ms/step - accuracy: 0.0062 - loss: 5.4035
Epoch 1: val_accuracy improved from None to 0.00364, saving model to ../models/best_transformer_v3.keras

Epoch 1: finished saving model to ../models/best_transformer_v3.keras
238/238 ━━━━━━━━━━━━━━━━━━━━ 147s 585ms/step - accuracy: 0.0062 - loss: 5.4035 - val_accuracy: 0.0036 - val_loss: 5.3339 - learning_rate: 5.0000e-04
Epoch 2/60
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 705ms/step - accuracy: 0.0072 - loss: 5.3017
Epoch 2: val_accuracy improved from 0.00364 to 0.01091, saving model to ../models/best_transformer_v3.keras

Epoch 2: finished saving model to ../models/best_transformer_v3.keras
238/238 ━━━━━━━━━━━━━━━━━━━━ 171s 716ms/step - accuracy: 0.0072 - loss: 5.3017 - val_accuracy: 0.0109 - val_loss: 5.2167 - learning_rate: 5.0000e-04
Epoch 3/60
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 683ms/step - accuracy: 0.0162 - loss: 5.1374
Epoch 3: val_accuracy improved from 0.01091 to 0.03273, saving model to ../models/b

In [13]:
# ============================================
# Evaluate Transformer V3
# ============================================

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)

print("="*40)
print(f"Transformer V3 Test Accuracy : {test_acc*100:.2f}%")
print(f"Transformer V3 Test Loss     : {test_loss:.4f}")
print("="*40)

16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 280ms/step - accuracy: 0.7290 - loss: 1.3321
Transformer V3 Test Accuracy : 72.90%
Transformer V3 Test Loss     : 1.3321
